In [1]:
import pandas as pd
import numpy as np
import os
import torch
from transformers import AutoTokenizer, AutoModel

# Φορτώνουμε το dataset
data_path = os.path.expanduser("~/igbert-embedding-analysis/data/processed/sequences_balanced.csv")
df = pd.read_csv(data_path)

print(f"Dataset: {df.shape}")
print(f"\nΠρώτες 3 γραμμές:")
print(df.head(3))
print(f"\nCPU ή GPU:")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Χρησιμοποιούμε: {device}")

Dataset: (4500, 4)

Πρώτες 3 γραμμές:
                               sequence_alignment_aa       v_call v_family  \
0  ASVKVSCKVSGYTLTELSLHWVRQAPGKGLEWMGGFDPEDGKTIYA...  IGHV1-24*01    IGHV1   
1  SVKVSCKASGGTFSSYAISWVRQAPGQGLEWMGGIIPIFGTANYAQ...  IGHV1-69*13    IGHV1   
2  ASVKVSCKASGYTFTGYYMHWVRQAPGQGLEWMGWINPNSGGTNYA...   IGHV1-2*02    IGHV1   

  isotype  
0    IGHA  
1    IGHA  
2    IGHA  

CPU ή GPU:
Χρησιμοποιούμε: cpu


In [2]:
# Φορτώνουμε το IgBert από το HuggingFace
print("Κατεβάζω το IgBert...")
tokenizer = AutoTokenizer.from_pretrained("Exscientia/IgBert")
model = AutoModel.from_pretrained("Exscientia/IgBert")
model = model.to(device)
model.eval()
print("Έτοιμο!")
print(f"\nΑρχιτεκτονική:")
print(f"Layers: {model.config.num_hidden_layers}")
print(f"Hidden size: {model.config.hidden_size}")

Κατεβάζω το IgBert...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt:   0%|          | 0.00/81.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/686 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.68G [00:00<?, ?B/s]

Some weights of BertModel were not initialized from the model checkpoint at Exscientia/IgBert and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Έτοιμο!

Αρχιτεκτονική:
Layers: 30
Hidden size: 1024


In [3]:
def get_embeddings(sequences, tokenizer, model, device, layer=30, batch_size=32):
    """
    Παίρνει embeddings από συγκεκριμένο layer του IgBert.
    Κάθε αλληλουχία → mean pooling των token embeddings → διάνυσμα 1024
    """
    all_embeddings = []
    
    for i in range(0, len(sequences), batch_size):
        batch = sequences[i:i+batch_size]
        
        # Tokenization: μετατρέπουμε αλληλουχίες σε αριθμούς
        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=256
        ).to(device)
        
        # Forward pass χωρίς gradient computation (είμαστε σε inference mode)
        with torch.no_grad():
            outputs = model(**inputs, output_hidden_states=True)
        
        # Παίρνουμε το συγκεκριμένο layer
        hidden_state = outputs.hidden_states[layer]
        
        # Mean pooling: από (batch, seq_len, 1024) → (batch, 1024)
        attention_mask = inputs['attention_mask'].unsqueeze(-1)
        embeddings = (hidden_state * attention_mask).sum(1) / attention_mask.sum(1)
        
        all_embeddings.append(embeddings.cpu().numpy())
        
        if (i // batch_size) % 10 == 0:
            print(f"  {i}/{len(sequences)} sequences...")
    
    return np.vstack(all_embeddings)

print("Συνάρτηση έτοιμη.")

Συνάρτηση έτοιμη.


In [4]:
sequences = df['sequence_alignment_aa'].tolist()

print("Παράγω embeddings από layer 30 (τελευταίο)...")
embeddings_l30 = get_embeddings(sequences, tokenizer, model, device, layer=30)

print("\nΠαράγω embeddings από layer 15 (μεσαίο)...")
embeddings_l15 = get_embeddings(sequences, tokenizer, model, device, layer=15)

print(f"\nΑποτελέσματα:")
print(f"Layer 30 shape: {embeddings_l30.shape}")
print(f"Layer 15 shape: {embeddings_l15.shape}")

Παράγω embeddings από layer 30 (τελευταίο)...
  0/4500 sequences...
  320/4500 sequences...
  640/4500 sequences...
  960/4500 sequences...
  1280/4500 sequences...
  1600/4500 sequences...
  1920/4500 sequences...
  2240/4500 sequences...
  2560/4500 sequences...
  2880/4500 sequences...
  3200/4500 sequences...
  3520/4500 sequences...
  3840/4500 sequences...
  4160/4500 sequences...
  4480/4500 sequences...

Παράγω embeddings από layer 15 (μεσαίο)...
  0/4500 sequences...
  320/4500 sequences...
  640/4500 sequences...
  960/4500 sequences...
  1280/4500 sequences...
  1600/4500 sequences...
  1920/4500 sequences...
  2240/4500 sequences...
  2560/4500 sequences...
  2880/4500 sequences...
  3200/4500 sequences...
  3520/4500 sequences...
  3840/4500 sequences...
  4160/4500 sequences...
  4480/4500 sequences...

Αποτελέσματα:
Layer 30 shape: (4500, 1024)
Layer 15 shape: (4500, 1024)
